# 1.1 Empirical Risk Minimization

We cannot measure the true error, so we minimize the error we can see: the average loss on the training sample.

Loss functions turn mistakes into numbers, empirical risk averages them on the sample, and ERM chooses the hypothesis with the smallest average. The same act of minimizing observed loss also creates the overfitting risk that later theory has to bound.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)


def zero_one_loss(pred, y):
    pred = np.asarray(pred)
    y = np.asarray(y)
    return (pred != y).astype(float)


def empirical_risk(h, X, y, loss=zero_one_loss):
    pred = h(np.asarray(X))
    losses = loss(pred, np.asarray(y))
    return float(np.mean(losses))


def threshold_rule(t):
    def h(X):
        X = np.asarray(X)
        return (X > t).astype(int)
    return h


def memorize_rule(train_x, train_y, default=0):
    table = {float(x): int(label) for x, label in zip(train_x, train_y)}

    def h(X):
        X = np.asarray(X)
        values = [table.get(float(x), default) for x in X]
        return np.asarray(values, dtype=int)
    return h


def true_rule(X):
    X = np.asarray(X)
    return (X > 2.5).astype(int)


def holdout_grid(n=200):
    X = np.linspace(1.0, 5.0, n)
    y = true_rule(X)
    return X, y


def best_threshold(X, y, thresholds):
    risks = []
    for t in thresholds:
        risk = empirical_risk(threshold_rule(t), X, y)
        risks.append(risk)
    index = int(np.argmin(risks))
    return float(thresholds[index]), float(risks[index]), np.asarray(risks)


def rung_record(name, X, y, thresholds, include_memorizer=False):
    t, train_risk, risks = best_threshold(X, y, thresholds)
    h = threshold_rule(t)
    if include_memorizer:
        mem = memorize_rule(X, y)
        mem_risk = empirical_risk(mem, X, y)
        if mem_risk <= train_risk:
            h = mem
            train_risk = mem_risk
            t = np.nan
    Hx, Hy = holdout_grid()
    holdout_risk = empirical_risk(h, Hx, Hy)
    return {
        "name": name,
        "X": np.asarray(X, dtype=float),
        "y": np.asarray(y, dtype=int),
        "thresholds": np.asarray(thresholds, dtype=float),
        "best_t": t,
        "train_risk": train_risk,
        "holdout_risk": holdout_risk,
        "gap": holdout_risk - train_risk,
        "risks": risks,
        "memorizer": include_memorizer,
        "h": h,
    }

## The concept, built once (D1)

For a sample $S=\{(x_i,y_i)\}_{i=1}^m$, empirical risk is

$$R_S(h)=rac{1}{m}\sum_{i=1}^{m}\ell(h(x_i),y_i).$$

The lesson's hand case has five ordered labels and three thresholds. Building ERM means writing one risk function, evaluating every candidate in $H$, and choosing the smallest value.

In [ ]:
X_d1 = np.array([1, 2, 3, 4, 5], dtype=float)
y_d1 = np.array([0, 0, 1, 1, 1], dtype=int)
lesson_thresholds = np.array([1.5, 2.5, 3.5], dtype=float)
lesson_risks = []
for t in lesson_thresholds:
    risk = empirical_risk(threshold_rule(t), X_d1, y_d1)
    lesson_risks.append(risk)
lesson_risks = np.asarray(lesson_risks)
print(dict(zip(lesson_thresholds, lesson_risks)))

The lesson numbers are $t=1.5	o1/5$, $t=2.5	o0/5$, and $t=3.5	o1/5$. These asserts make the notebook prove that the implemented ERM matches the lesson arithmetic exactly.

In [ ]:
assert np.allclose(lesson_risks, [1 / 5, 0 / 5, 1 / 5])
best_t, best_risk, _ = best_threshold(X_d1, y_d1, lesson_thresholds)
assert best_t == 2.5
assert best_risk == 0.0
print(f"ERM selects t={best_t} with R_S={best_risk:.2f}")

## The dataset ladder

The ladder is a theory-capacity ladder, not a shared data ladder: D1 is the hand-verifiable case, D2 raises threshold resolution, D3 adds a noisy Bayes-floor illustration, D4 increases sample size under the same threshold class, and D5 adds a memorizing hypothesis for each observed point.

In [ ]:
rungs = []
rungs.append(rung_record("D1 hand thresholds", X_d1, y_d1, lesson_thresholds))
thresholds_d2 = np.linspace(1.5, 4.5, 7)
rungs.append(rung_record("D2 seven thresholds", X_d1, y_d1, thresholds_d2))
X_d3 = np.linspace(1.0, 5.0, 50)
y_d3 = true_rule(X_d3)
flip_index = rng.choice(len(y_d3), size=5, replace=False)
y_d3[flip_index] = 1 - y_d3[flip_index]
rungs.append(rung_record("D3 ten percent noisy labels", X_d3, y_d3, thresholds_d2))
X_d4 = np.linspace(1.0, 5.0, 200)
y_d4 = true_rule(X_d4)
rungs.append(rung_record("D4 n=200 fixed class", X_d4, y_d4, thresholds_d2))
X_d5 = np.linspace(1.0, 5.0, 80)
y_d5 = true_rule(X_d5)
flip_index = rng.choice(len(y_d5), size=16, replace=False)
y_d5[flip_index] = 1 - y_d5[flip_index]
rungs.append(rung_record("D5 threshold plus memorizer", X_d5, y_d5, thresholds_d2, include_memorizer=True))

for rung in rungs:
    print(rung["name"])
    print("  n=", len(rung["X"]))
    print("  class size=", len(rung["thresholds"]) + int(rung["memorizer"]))
    print("  sample=", list(zip(rung["X"][:5], rung["y"][:5])))

In [ ]:
summary = []
for rung in rungs:
    summary.append((rung["name"], rung["train_risk"], rung["holdout_risk"], rung["gap"]))
print("rung | train risk | holdout risk | gap")
for name, train_risk, holdout_risk, gap in summary:
    print(f"{name}: {train_risk:.3f} | {holdout_risk:.3f} | {gap:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], rungs):
    order = np.argsort(rung["X"])
    X_sorted = rung["X"][order]
    y_sorted = rung["y"][order]
    pred = rung["h"](X_sorted)
    ax.scatter(X_sorted, y_sorted, label="labels", s=18)
    ax.step(X_sorted, pred, where="mid", label="ERM prediction")
    ax.set_title(rung["name"])
    ax.set_ylim(-0.2, 1.2)
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(loc="lower right")
capacities = [len(rung["thresholds"]) + int(rung["memorizer"]) for rung in rungs]
gaps = [rung["gap"] for rung in rungs]
axes[1, 0].plot(capacities, gaps, marker="o")
axes[1, 0].set_xlabel("hypothesis-class capacity")
axes[1, 0].set_ylabel("holdout minus empirical risk")
axes[1, 0].set_title("Payoff curve: generalization gap")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on the hardest rung

Pitfall: zero training error is a warning, not a trophy. On D5 the memorizer can make $R_S=0$ by encoding the sample, but the held-out risk exposes that this is not the same as learning the population rule.

In [ ]:
d5 = rungs[-1]
mem = memorize_rule(d5["X"], d5["y"])
Hx, Hy = holdout_grid()
wrong_train = empirical_risk(mem, d5["X"], d5["y"])
wrong_holdout = empirical_risk(mem, Hx, Hy)
fixed_t, fixed_train, _ = best_threshold(d5["X"], d5["y"], d5["thresholds"])
fixed_h = threshold_rule(fixed_t)
fixed_holdout = empirical_risk(fixed_h, Hx, Hy)
print(f"wrong memorizer: train={wrong_train:.3f}, holdout={wrong_holdout:.3f}")
print(f"fixed threshold t={fixed_t:.2f}: train={fixed_train:.3f}, holdout={fixed_holdout:.3f}")

## Evaluate it + Practice

- Metric: empirical-vs-holdout risk gap.
- No-skill baseline: choose the first threshold without minimizing risk.
- Cheap sanity check: recompute the D1 mistake counts by hand.
- Ablation: remove the holdout check and D5 looks falsely perfect.
- Failure signals: zero training loss with a rising holdout gap or many tied ERM rules.

### Practice prompts

- Add a fourth threshold to D1 and recompute the ERM choice.

- Change the D5 noise rate and predict how the memorizer gap changes.

- Add a penalty of 0.02 per extra hypothesis and select again.